# Lab 2 — Six tracks, two favourites
### EP0418 · Topic 2 · 2 practical hours (Week 2)

**Nothing is decided today.** Your project track is chosen with your group in
Week 9, when the project starts, with eight labs behind you. Today you taste all
six, so that the choice is an informed one when it comes.

---
## What you do today

1. Run the six baselines below (Task 1), then open the starter packs of the two
   tracks you like most and run them in full.
2. For each, note its **built-in baseline** — the number a project would have to beat.
3. Write down your two favourites and why. The labs from here to Lab 6 run on
   shared data; the pre-lab tasks ask you to look at the dataset you liked most.

Every track is graded against the same rubric. The domain changes; the assessed
outcomes do not.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). Then:

1. In its chat box, click the **sliders icon** next to **+** and choose **Configure**. The Notebook settings window opens.
2. Click **Add new custom instruction**. Name it **EP0418**, paste the text below into Instructions, click **Add to notebook**, then **Apply**.
3. Click the sliders icon again and choose **EP0418**, so the tick moves from Default to EP0418. That is the mode you work in.

Colab keeps the instruction with this notebook. Do the same once in each new notebook; the text is always in this cell.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

**Why these seven lines.** Each one stops a mistake that AI assistants make with data, and that you would not notice until a check failed:

| Line | The mistake it prevents |
|---|---|
| Explain in plain words | Code you cannot account for. You will be asked what each step does to your data. |
| Plan first | Ten lines of code for the wrong task. The plan is quicker to correct than the code. |
| Never drop rows unless asked | `dropna()` quietly removing a third of a sensor file. |
| Row count and range before and after | The only cheap way to see what a step really did. |
| -999 is missing, not a number | An average of humidity that comes out negative. |
| Never touch `check.` lines | A test rewritten so that it passes. The checks are the contract. |
| Say so if unsure | A confident guess that reads exactly like a fact. |

**See it working (1 minute).** With EP0418 selected, ask Gemini:

> Write code to remove the rows of df where humidity_pct is -999.

It should warn you that this removes rows before it writes anything, treat -999 as a missing reading rather than a value, and explain the code in plain words. Now switch the tick to **Default**, ask the same thing in a new chat, and compare. Switch back to **EP0418**.

**Make it yours.** Click the sliders icon, choose Configure, and open EP0418 with the pencil icon. Add one line of your own, for example *Keep answers under ten lines* or *Always name the column you are using*, and Apply. Ask the same question again and see your line take effect. The seven course lines stay; yours go underneath.

## The six tracks

| Track | Data | Macro lesson | Baseline to beat |
|---|---|---|---|
| 1 Smart Money | STI ETF daily bars | Beating buy-and-hold is hard | Buy-and-hold |
| 2 Smart Building | Classroom IoT sensors | Anomalies, stuck sensors, drift | Fixed threshold |
| 3 Smart Farm | Real NEA rainfall + PSI | Forecasting into a decision | "Yesterday's value" |
| 4 Smart City | Live taxi availability | Seasonality and decomposition | Seasonal-naive |
| 5 Smart Self | Daily wearable data | Personal patterns + privacy | Rolling 7-day mean |
| 6 Free-form | Yours | Whatever your problem needs | You define it |

> **Track 1 is paper-trading only.** No real money, no live broker. Enforced by
> the rubric; live deployment earns zero additional marks.

> **Track 5 handles personal data.** If you use your own export, anonymise it.
> Using the sample data instead is a legitimate choice and costs you no marks.

> **Track 6 must pass the feasibility diagnostic** in its notebook *and* get
> tutor sign-off before you commit.

## Task 1 — Run all six

The cells below run the baseline of every track in one place; each fetches its
own data. Run them in order. Each section ends with a line beginning
`RECORD THIS:`; that is the number for the table at the end of this task. Do not
edit anything. You are looking, not building.

In [ ]:
import os
import pandas as pd, numpy as np
from ep0418 import show

PACK = {"smart_money_ohlcv.csv": "track1_smart_money", "smart_building_sensors.csv": "track2_smart_building",
        "sg_rainfall_daily.csv": "track3_smart_farm",
        "sg_taxi_hourly.csv": "track4_smart_city", "smart_self_daily.csv": "track5_smart_self",
        "example_data.csv": "track6_free_form"}
RAW = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/"

def find(name):
    # beside the notebook (or unzipped), inside a track folder, or fetched from the course repository
    for cand in [name] + [os.path.join(d, name) for d in os.listdir(".") if os.path.isdir(d)]:
        if os.path.exists(cand):
            return cand
    try:
        import urllib.request
        urllib.request.urlretrieve(RAW + PACK[name] + "/" + name, name)
    except Exception:
        from google.colab import files; files.upload()
    return name

RECORD = {}
def record(track, label, value):
    RECORD[track] = (label, value)
    print(f"\nRECORD THIS:  {label} = {value}")

## Track 1 — Smart Money (STI ETF daily bars)

In [ ]:
df = pd.read_csv(find("smart_money_ohlcv.csv"), parse_dates=["Date"]).sort_values("Date")
print(f"rows {len(df):,}   period {df.Date.min().date()} -> {df.Date.max().date()}   cadence: trading days")
bh_total = df.Close.iloc[-1] / df.Close.iloc[0] - 1
print(f"buy-and-hold total return: {bh_total:+.1%}   (buy on day 1, sell on the last day, no trading)")
record("1 Smart Money", "buy-and-hold return", f"{bh_total:+.1%}")
show.series(df, col="Close", title="Track 1: closing price, one point per trading day")

## Track 2 — Smart Building (classroom sensors, every 5 minutes)

In [ ]:
b = pd.read_csv(find("smart_building_sensors.csv"), parse_dates=["timestamp"])
print(f"rows {len(b):,}   period {b.timestamp.min()} -> {b.timestamp.max()}   cadence: 5 minutes")
alert = b.co2_ppm > 1000
print(f"fixed-threshold alerts (CO2 > 1000 ppm): {alert.sum()} of {len(b):,} readings; "
      f"{(alert & (b.occupancy == 0)).sum()} fired while the room was empty")
record("2 Smart Building", "alerts from the fixed threshold", int(alert.sum()))
show.series(b, col="co2_ppm", title="Track 2: CO2 every 5 minutes, with the alert threshold", hline=1000)

## Track 3 — Smart Farm (NEA daily rainfall)

In [ ]:
r = pd.read_csv(find("sg_rainfall_daily.csv"), parse_dates=["date"]).sort_values("date")
print(f"rows {len(r)}   period {r.date.min().date()} -> {r.date.max().date()}   cadence: daily")
needs = r.rain_mm_mean < 1.0                       # the field needs water
yesterday = needs.shift(1)
m = yesterday.notna()
correct = (yesterday[m].astype(bool) == needs[m]).mean()
print(f"policy 'water if yesterday needed water': right on {correct:.1%} of days "
      f"({int((needs[m] & ~yesterday[m].astype(bool)).sum())} missed waterings, "
      f"{int((~needs[m] & yesterday[m].astype(bool)).sum())} wasted)")
record("3 Smart Farm", "days 'yesterday's value' was right", f"{correct:.1%}")
show.series(r, col="rain_mm_mean", title="Track 3: island-wide mean rainfall per day", hline=1.0)

## Track 4 — Smart City (taxi availability, about hourly)

In [ ]:
t = pd.read_csv(find("sg_taxi_hourly.csv"), parse_dates=["timestamp"])
print(f"rows {len(t):,}   period {t.timestamp.min()} -> {t.timestamp.max()}   cadence: about hourly (irregular)")
s = t.set_index("timestamp").taxi_count.resample("h").mean()
sn = s.shift(168); m = s.notna() & sn.notna()
mae = (s - sn)[m].abs().mean()
print(f"seasonal-naive MAE (same hour last week): {mae:,.1f} taxis   (mean count {s.mean():,.0f})")
record("4 Smart City", "seasonal-naive MAE, taxis", round(float(mae), 1))
show.series(s, title="Track 4: taxis available, hourly")

## Track 5 — Smart Self (wearable, one row per day)

In [ ]:
w = pd.read_csv(find("smart_self_daily.csv"), parse_dates=["date"]).sort_values("date")
print(f"rows {len(w)}   period {w.date.min().date()} -> {w.date.max().date()}   cadence: daily "
      f"({(w.date.max() - w.date.min()).days + 1 - len(w)} dates absent)")
d = w.set_index("date").asfreq("D")
pred = d.steps.rolling(7).mean().shift(1); m = pred.notna() & d.steps.notna()
mae = (d.steps[m] - pred[m]).abs().mean()
print(f"rolling 7-day mean baseline: MAE {mae:,.0f} steps   (mean daily steps {d.steps.mean():,.0f})")
record("5 Smart Self", "rolling 7-day MAE, steps", int(round(mae)))
show.series(d.steps, title="Track 5: daily steps (gaps are days with no row)")

## Track 6 — Free-form (the example file; yours will differ)

In [ ]:
e = pd.read_csv(find("example_data.csv"), parse_dates=["timestamp"]).sort_values("timestamp")
print(f"rows {len(e)}   period {e.timestamp.min().date()} -> {e.timestamp.max().date()}")
x = e.set_index("timestamp")["value"]
for name, p in [("naive (last value)", x.shift(1)), ("seasonal-naive (7 back)", x.shift(7))]:
    m = p.notna() & x.notna()
    print(f"{name:26s} MAE {(x[m] - p[m]).abs().mean():.3f}")
m = x.shift(1).notna()
record("6 Free-form", "naive MAE on the example file", round(float((x[m] - x.shift(1)[m]).abs().mean()), 3))
show.series(x, title="Track 6: the example file")

## The six numbers for your sheet

Copy this table into the Lab 2 sheet. Then answer the question that actually
shapes your shortlist: which of these would be hardest to beat, and why?

In [ ]:
print(f"{'Track':18s} {'Baseline':40s} {'Number'}")
for track, (label, value) in RECORD.items():
    print(f"{track:18s} {label:40s} {value}")

Fill this in from the `RECORD THIS` lines:

| Track | Rows | What period | Baseline number | Would I enjoy 14 weeks of this? |
|---|---|---|---|---|
| 1 Smart Money | | | | |
| 2 Smart Building | | | | |
| 3 Smart Farm | | | | |
| 4 Smart City | | | | |
| 5 Smart Self | | | | |
| 6 Free-form | | | | |

**The six starter notebooks, one per track.** Open the two you like most.

| Track | Starter notebook (opens in Colab; then File > Save a copy in Drive) |
|---|---|
| 1 Smart Money | [Track1_Smart_Money_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track1_smart_money/Track1_Smart_Money_Starter.ipynb) |
| 2 Smart Building | [Track2_Smart_Building_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track2_smart_building/Track2_Smart_Building_Starter.ipynb) |
| 3 Smart Farm | [Track3_Smart_Farm_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track3_smart_farm/Track3_Smart_Farm_Starter.ipynb) |
| 4 Smart City | [Track4_Smart_City_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track4_smart_city/Track4_Smart_City_Starter.ipynb) |
| 5 Smart Self | [Track5_Smart_Self_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track5_smart_self/Track5_Smart_Self_Starter.ipynb) |
| 6 Free-form | [Track6_Free_Form_Starter.ipynb](https://colab.research.google.com/github/StanleySP-eng/EP0418-labs/blob/main/starter_packs/track6_free_form/Track6_Free_Form_Starter.ipynb) |

## Task 2 — The two questions that actually decide it

**1. How hard is the baseline?**
Some are much harder than they look. Buy-and-hold returned +99.4% over the sample
period — beating that after costs is genuinely difficult. A fixed CO₂ threshold,
by contrast, is easy to beat once you handle drift.

Neither is the "right" choice. But choose knowing which you picked.

**2. Where will your automated response go?**
Every project must trigger something real. Who receives it? If you
cannot name a plausible recipient for a track, that is a warning sign.

## Task 3 — Two favourites

In the cell below, write the two tracks you liked most and why. No commitment:
this is the shortlist you bring to your group in Week 9.

**Favourite 1:** *(track · why · its baseline number · who would receive the response)*

**Favourite 2:** *(track · why · its baseline number · who would receive the response)*

**The one I would find hardest, and why:**

---
## Read before you run

An assistant was asked how many days the Smart Self file covers. It wrote the code below.

**Do not run it yet.** Read it, answer the two questions below,
and only then remove the `#` and run it to see whether you were right.

What does the last line actually count, and why is that not the number of days covered?

Write down the number it would print, and the number it should print.

<details><summary>Hint 1: where to look</summary>

Count what `len()` counts. Is it rows, or days? Compare the first and last date with the number of rows.

</details>
<details><summary>Hint 2: closer to the answer</summary>

The file runs 20 January to 18 July, 180 calendar days, but has 177 rows. Three dates are absent entirely.

</details>

**After you have written your answer, and not before:** select the cell, click the
pencil-and-spark icon at its top right and choose **Explain code**. Did Gemini notice
the fault? It often does not. That is the point: an explanation that reads well is
not evidence the code is right.

**To fix it:** choose **Transform code** on the same cell and say what to change, in
plain words. Colab shows the change as a diff before it is applied. Read the diff. If
it touches a `check.` line, reject it.

In [ ]:
# --- AI-generated. Read it first. ---
import pandas as pd, os, urllib.request
if not os.path.exists("smart_self_daily.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/smart_self_daily.csv", "smart_self_daily.csv")

# df = pd.read_csv("smart_self_daily.csv")
# print("from", df.date.min(), "to", df.date.max())
# print("days covered:", len(df))

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Stay in EP0418 mode and paste this into the Gemini chat:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: what a baseline is and why every track has one; which baseline belongs to which track; what makes a baseline hard to beat; the difference between a static file and a live feed; what the Track 6 feasibility diagnostic checks. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

- Two favourites written down above, with reasons.
- **Mistake caught** (below): anything that surprised you in a pack you ran, and
  how you noticed it.

**Mistake caught:** *(what I expected · what was there · how I found it)*